In [ ]:
# ============================================================
# PARAMETERS (papermill injects these)
# ============================================================

config_path = None
run_dir = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import json
import yaml
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from scripts.set_seed import set_seed

from src.model_factory import build_model

from src.embedding_registry import get_embedding_dir

from src.embeddings import load_single_embeddings_from_manifest

from src.training import (
    train_classifier,
    print_final_training_summary,
    summarize_final_in_sample_metrics,
    final_in_sample_classification_table,
    plot_final_training_history,
    plot_final_macro_metrics,
    plot_final_confusion_matrix,
    plot_final_roc_curves,
    evaluate_split
)

In [ ]:
# ============================================================
# LOAD CONFIG, SEED, AND MODEL
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

RUN_DIR = Path(run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg["experiment"]["seed"])

model = build_model(cfg)

print(json.dumps(cfg, indent=2))

In [ ]:
# ============================================================
# LOAD EMBEDDINGS
# ============================================================

evaluation_mode = cfg["evaluation"]["mode"]

print(f"Evaluation mode: {evaluation_mode}")

embedding_dir = get_embedding_dir(cfg)

embeddings_dict = load_single_embeddings_from_manifest(embedding_dir, init_embedder=False)

In [ ]:
# ============================================================
# FINAL MODEL TRAINING
# ============================================================

# Prepare all-data loaders 
train_ds = TensorDataset(embeddings_dict['train_embeddings'], embeddings_dict['train_labels'])
val_ds = TensorDataset(embeddings_dict['val_embeddings'], embeddings_dict['val_labels'])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# Train model
final_model = model().to(device)

history = train_classifier(
    final_model,
    train_loader,
    val_loader,
    num_epochs=50,
    device=device,
    patience=50,
    criterion=None,
    checkpoint_path="model_best_epoch.pt"
)

In [ ]:
# Disclaimer
print("⚠️ The in-sample macro F1 and AUC are expected to be high (possibly near-perfect) since there's no held-out data — these numbers should not be interpreted as generalisation performance.")

# Display tables
print_final_training_summary(history_final, save_path=RUN_DIR / "final_training")
summarize_final_in_sample_metrics(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")
final_in_sample_classification_table(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")

# Display plots
plot_final_training_history(history_final, save_path=RUN_DIR / "final_training")
plot_final_macro_metrics(history_final, save_path=RUN_DIR / "final_training")
plot_final_confusion_matrix(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")
plot_final_roc_curves(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

torch.save(
    final_model.state_dict(),
    RUN_DIR / "final_model.pt",
)

print("Final model saved.")

In [ ]:
# ============================================================
# SAVE FINAL METADATA
# ============================================================

final_metadata = {
    "experiment_name": cfg["experiment"]["name"],
    "evaluation_mode": evaluation_mode,
    "best_config": BEST_CFG,
}

with open(
    RUN_DIR / "training_metadata.json",
    "w",
) as f:

    json.dump(final_metadata, f, indent=2)

print("Training complete.")